In [ ]:
# Cell 1 — Install (restart kernel after this cell)
import subprocess
subprocess.run(['pip', 'uninstall', 'unsloth', 'unsloth_zoo', 'torchao', 'trl', 'transformers', 'bitsandbytes', '-y'])
subprocess.run(['pip', 'install', '-q',
    'transformers==4.46.3',
    'peft==0.13.2',
    'accelerate==1.1.1',
    'datasets',
    'seqeval',
    'tqdm',
    'faiss-cpu',
    'sentence-transformers',
])
print('Done. Restart kernel now, then run all cells below.')

In [1]:
# Cell 2 — Find adapter path (auto-discovers from notebook output)
import os

ADAPTER_PATH = None
for root, dirs, files in os.walk('/kaggle/input'):
    if 'adapter_config.json' in files:
        ADAPTER_PATH = root
        break


if ADAPTER_PATH is None:
    raise FileNotFoundError(
        'adapter_config.json not found under /kaggle/input.\n'
        'Add the P-6 notebook output via: Add data → Your work → P-6: SRICL (All data) → Output'
    )
print(f'Adapter found at: {ADAPTER_PATH}')
print('Files:', os.listdir(ADAPTER_PATH))

Adapter found at: /kaggle/input/notebooks/acme105/p-6-sricl-all-data/checkpoints/checkpoint-2172
Files: ['adapter_model.safetensors', 'merges.txt', 'trainer_state.json', 'training_args.bin', 'adapter_config.json', 'README.md', 'tokenizer.json', 'vocab.json', 'tokenizer_config.json', 'scheduler.pt', 'special_tokens_map.json', 'optimizer.pt', 'rng_state.pth', 'added_tokens.json']


In [2]:
# Cell 3 — Imports
os.environ['CUDA_VISIBLE_DEVICES'] = '0'
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'

import random
import numpy as np
import pandas as pd
import torch
from tqdm import tqdm
from datasets import load_dataset
from seqeval.metrics import f1_score, classification_report
from sentence_transformers import SentenceTransformer
import faiss
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

random.seed(42)
torch.manual_seed(42)
print('Imports OK')
print(f'PyTorch: {torch.__version__}')
print(f'CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'Free: {torch.cuda.mem_get_info()[0]/1e9:.1f} GB')

2026-05-18 16:50:25.149795: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1779123025.352999     116 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1779123025.409101     116 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1779123025.909330     116 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1779123025.909367     116 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1779123025.909370     116 computation_placer.cc:177] computation placer alr

Imports OK
PyTorch: 2.10.0+cu128
CUDA: True
GPU: Tesla T4
Free: 15.5 GB


In [3]:
# Cell 4 — Load all 6 datasets
say = load_dataset('jjzha/sayfullina')
skl = load_dataset('jjzha/skillspan')
grn = load_dataset('jjzha/green')
gnh = load_dataset('jjzha/gnehm')
fij = load_dataset('jjzha/fijo')
kom = load_dataset('jjzha/kompetencer')

print('Sayfullina   train/test:', len(say['train']), len(say['test']))
print('SkillSpan    train/test:', len(skl['train']), len(skl['test']))
print('GREEN        train/test:', len(grn['train']), len(grn['test']))
print('GNEHM        train/test:', len(gnh['train']), len(gnh['test']))
print('FIJO         train/test:', len(fij['train']), len(fij['test']))
print('Kompetencer  train/test:', len(kom['train']), len(kom['test']))

README.md: 0.00B [00:00, ?B/s]

train.json: 0.00B [00:00, ?B/s]

dev.json: 0.00B [00:00, ?B/s]

test.json: 0.00B [00:00, ?B/s]

Generating train split:   0%|          | 0/3705 [00:00<?, ? examples/s]

...

test.json: 0.00B [00:00, ?B/s]

Generating train split:   0%|          | 0/778 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/346 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/262 [00:00<?, ? examples/s]

Sayfullina   train/test: 3705 1851
SkillSpan    train/test: 4800 3569
GREEN        train/test: 8669 335
GNEHM        train/test: 19889 2557
FIJO         train/test: 399 50
Kompetencer  train/test: 778 262


In [4]:
# Cell 5 — Tag normalisation + BIO helpers
def normalise_tags(tags):
    result = []
    for tag in tags:
        if tag in ('B', 'B-SOFT', 'B-ICT', 'B-SKILL',
                   'B-PENSEE', 'B-RESULTATS', 'B-RELATIONNEL', 'B-PERSONNEL'):
            result.append('B-SKILL')
        elif tag in ('I', 'I-SOFT', 'I-ICT', 'I-SKILL',
                     'I-PENSEE', 'I-RESULTATS', 'I-RELATIONNEL', 'I-PERSONNEL'):
            result.append('I-SKILL')
        else:
            result.append('O')
    return result


def get_tokens_and_tags(example):
    return example['tokens'], normalise_tags(example['tags_skill'])


def to_bracket_format(tokens, tags):
    result = []
    i = 0
    while i < len(tokens):
        if tags[i] == 'B-SKILL':
            span = [tokens[i]]
            j = i + 1
            while j < len(tokens) and tags[j] == 'I-SKILL':
                span.append(tokens[j])
                j += 1
            result.append(f"@@{' '.join(span)}##")
            i = j
        else:
            result.append(tokens[i])
            i += 1
    return ' '.join(result)


def bio_from_bracket(tokens, output_text):
    tags = ['O'] * len(tokens)
    text = output_text
    spans = []
    while '@@' in text and '##' in text:
        start = text.find('@@')
        end   = text.find('##', start)
        if start == -1 or end == -1:
            break
        spans.append(text[start+2:end].strip())
        text = text[end+2:]
    for span in spans:
        span_tokens = span.split()
        n = len(span_tokens)
        for i in range(len(tokens) - n + 1):
            if tokens[i:i+n] == span_tokens:
                if all(tags[i+j] == 'O' for j in range(n)):
                    tags[i] = 'B-SKILL'
                    for j in range(1, n):
                        tags[i+j] = 'I-SKILL'
                break
    return tags


def spans_from_bio(tokens, tags):
    spans, current = [], []
    for token, tag in zip(tokens, tags):
        if tag == 'B-SKILL':
            if current:
                spans.append(' '.join(current))
            current = [token]
        elif tag == 'I-SKILL' and current:
            current.append(token)
        else:
            if current:
                spans.append(' '.join(current))
            current = []
    if current:
        spans.append(' '.join(current))
    return spans

print('Helpers defined')

Helpers defined


In [5]:
# Cell 6 — System prompt (must match training prompt exactly)
SYSTEM_ICL = (
    'You are a skill-span extractor specialising in job advertisement sentences. '
    'Your task is to identify ALL skill spans — both hard skills (tools, technologies, '
    'programming languages, domain knowledge) and soft skills (interpersonal, behavioural, '
    'personal competencies such as communication, leadership, teamwork). '
    'Sentences may be in any language and may be grammatically incomplete. '
    'Extract spans exactly as they appear, token-by-token. '
    'Wrap each skill span with @@...## brackets. '
    'Copy the sentence exactly with no other changes. '
    'If there are no skills, return the sentence unchanged. '
    'Output only the annotated sentence — no explanation, no commentary.'
)


def format_prompt(user_text):
    return (
        f'<|im_start|>system\n{SYSTEM_ICL}<|im_end|>\n'
        f'<|im_start|>user\n{user_text}<|im_end|>\n'
        f'<|im_start|>assistant\n'
    )

print('Prompt defined')

Prompt defined


In [6]:
# Cell 7 — Build RAG-1 index (combined training pool, all 6 datasets)
embedder = SentenceTransformer('all-MiniLM-L6-v2')

ALL_DATASETS = [
    ('sayfullina', say),
    ('skillspan',  skl),
    ('green',      grn),
    ('gnehm',      gnh),
    ('fijo',       fij),
    ('kompetencer',kom),
]

all_train_with_skills = []
for name, ds in ALL_DATASETS:
    count = 0
    for ex in ds['train']:
        tokens, tags = get_tokens_and_tags(ex)
        if any(t != 'O' for t in tags):
            all_train_with_skills.append({'tokens': tokens, 'tags': tags, 'source': name})
            count += 1
    print(f'{name}: {count} examples with skills')

print(f'\nTotal RAG-1 pool: {len(all_train_with_skills)}')

rag1_sentences = [' '.join(ex['tokens']) for ex in all_train_with_skills]
print('Embedding RAG-1 sentences...')
rag1_emb = embedder.encode(rag1_sentences, show_progress_bar=True, batch_size=64).astype('float32')
faiss.normalize_L2(rag1_emb)
rag1_index = faiss.IndexFlatIP(rag1_emb.shape[1])
rag1_index.add(rag1_emb)
print(f'RAG-1 index: {rag1_index.ntotal} entries')

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

sayfullina: 3701 examples with skills
skillspan: 1001 examples with skills
green: 5136 examples with skills
gnehm: 3408 examples with skills
fijo: 384 examples with skills
kompetencer: 174 examples with skills

Total RAG-1 pool: 13804
Embedding RAG-1 sentences...


Batches:   0%|          | 0/216 [00:00<?, ?it/s]

RAG-1 index: 13804 entries


In [7]:
# Cell 8 — Build RAG-2 ESCO index
ESCO_PATH = None
for root, _, files in os.walk('/kaggle/input'):
    for f in files:
        if f == 'skills_en.csv':
            ESCO_PATH = os.path.join(root, f)
            break
    if ESCO_PATH:
        break

if ESCO_PATH is None:
    raise FileNotFoundError('skills_en.csv not found. Add the ESCO dataset via the Data panel.')
print(f'ESCO path: {ESCO_PATH}')

esco = pd.read_csv(ESCO_PATH)

def build_esco_text(row):
    parts = [str(row.get('preferredLabel', ''))]
    alt = str(row.get('altLabels', ''))
    if alt and alt != 'nan':
        parts.append(alt.replace('\n', ', '))
    desc = str(row.get('description', ''))
    if desc and desc != 'nan':
        parts.append(desc[:200])
    return ' | '.join(parts)

esco_texts  = [build_esco_text(row) for _, row in esco.iterrows()]
esco_labels = esco['preferredLabel'].tolist()

print('Embedding ESCO skills...')
rag2_emb = embedder.encode(esco_texts, show_progress_bar=True, batch_size=64).astype('float32')
faiss.normalize_L2(rag2_emb)
rag2_index = faiss.IndexFlatIP(rag2_emb.shape[1])
rag2_index.add(rag2_emb)
print(f'RAG-2 index: {rag2_index.ntotal} entries')

ESCO path: /kaggle/input/datasets/acme105/skills-en-esco/skills_en.csv
Embedding ESCO skills...


Batches:   0%|          | 0/219 [00:00<?, ?it/s]

RAG-2 index: 13960 entries


In [8]:
# Cell 9 — RAG retrieval + prompt builder
def retrieve_rag1(query_tokens, k=3):
    q_emb = embedder.encode([' '.join(query_tokens)]).astype('float32')
    faiss.normalize_L2(q_emb)
    _, indices = rag1_index.search(q_emb, k)
    return [(all_train_with_skills[i]['tokens'], all_train_with_skills[i]['tags'])
            for i in indices[0]]


def retrieve_rag2(candidate_spans, k=3):
    if not candidate_spans:
        return ''
    definitions = []
    for span in candidate_spans[:5]:
        q_emb = embedder.encode([span]).astype('float32')
        faiss.normalize_L2(q_emb)
        _, indices = rag2_index.search(q_emb, k)
        for idx in indices[0]:
            desc = str(esco.iloc[idx].get('description', ''))
            if desc and desc != 'nan':
                definitions.append(f'- {esco_labels[idx]}: {desc[:120]}')
    seen, unique = set(), []
    for d in definitions:
        if d not in seen:
            seen.add(d)
            unique.append(d)
    return '\n'.join(unique[:8])


def build_user_text(query_tokens, rag1_examples, esco_context):
    lines = []
    if rag1_examples:
        lines.append('Examples of skill annotation:')
        for ex_tokens, ex_tags in rag1_examples:
            lines.append(f'  Input:  {" ".join(ex_tokens)}')
            lines.append(f'  Output: {to_bracket_format(ex_tokens, ex_tags)}')
        lines.append('')
    if esco_context:
        lines.append('Relevant ESCO skill definitions:')
        lines.append(esco_context)
        lines.append('')
    lines.append(' '.join(query_tokens))
    return '\n'.join(lines)

print('RAG functions defined')

RAG functions defined


In [9]:
# Cell 10 — Load base model + saved LoRA adapter
import gc
gc.collect()
torch.cuda.empty_cache()
print(f'Free GPU: {torch.cuda.mem_get_info()[0]/1e9:.1f} GB')

BASE_MODEL = 'Qwen/Qwen2.5-3B-Instruct'

tokenizer = AutoTokenizer.from_pretrained(ADAPTER_PATH, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token

base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    torch_dtype=torch.float16,
    device_map={'': 0},
    trust_remote_code=True,
)
model = PeftModel.from_pretrained(base_model, ADAPTER_PATH)
model.eval()
print(f'Model loaded. GPU used: {torch.cuda.memory_allocated()/1e9:.1f} GB')

Free GPU: 15.4 GB


config.json:   0%|          | 0.00/661 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

model-00001-of-00002.safetensors:   0%|          | 0.00/3.97G [00:00<?, ?B/s]

model-00002-of-00002.safetensors:   0%|          | 0.00/2.20G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Model loaded. GPU used: 6.4 GB


In [10]:
# Cell 11 — BIO Verifier
def verify_bio(tags):
    prev_tag = 'O'
    for i, tag in enumerate(tags):
        if tag == 'O':
            prev_tag = 'O'
            continue
        prefix, label = tag.split('-', 1)
        if prefix == 'I':
            if prev_tag == 'O':
                return False, f'I-tag at position {i} ({tag}) follows O.'
            _, prev_label = prev_tag.split('-', 1)
            if prev_label != label:
                return False, f'I-tag at position {i} ({tag}) follows {prev_tag} — type mismatch.'
        prev_tag = tag
    return True, ''


def fix_bio(tags):
    fixed = list(tags)
    prev_tag = 'O'
    for i, tag in enumerate(fixed):
        if tag == 'O':
            prev_tag = 'O'
            continue
        prefix, label = tag.split('-', 1)
        if prefix == 'I':
            if prev_tag == 'O':
                fixed[i] = f'B-{label}'
            else:
                _, prev_label = prev_tag.split('-', 1)
                if prev_label != label:
                    fixed[i] = f'B-{label}'
        prev_tag = fixed[i]
    return fixed


def predict(tokens, max_retries=3):
    rag1_examples = retrieve_rag1(tokens, k=3)
    esco_context  = retrieve_rag2([' '.join(tokens)], k=3)
    prompt        = format_prompt(build_user_text(tokens, rag1_examples, esco_context))
    current_bio   = ['O'] * len(tokens)

    for _ in range(max_retries):
        inputs = tokenizer(
            prompt, return_tensors='pt', truncation=True, max_length=512
        ).to(model.device)

        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_new_tokens = 128,
                do_sample      = False,
                pad_token_id   = tokenizer.eos_token_id,
            )

        decoded = tokenizer.decode(
            outputs[0][inputs['input_ids'].shape[1]:],
            skip_special_tokens=True,
        )
        current_bio = bio_from_bracket(tokens, decoded)
        valid, error = verify_bio(current_bio)
        if valid:
            return current_bio

        prompt = (
            prompt + decoded + '<|im_end|>\n'
            '<|im_start|>user\n'
            f'BIO error: {error} Output the corrected annotated sentence only.<|im_end|>\n'
            '<|im_start|>assistant\n'
        )

    return fix_bio(current_bio)

print('Verifier + inference defined')

Verifier + inference defined


In [11]:
# Cell 12 — Evaluate all 6 datasets (200 examples each)
MAX_EXAMPLES = 500

EVAL_DATASETS = [
    ('Sayfullina',  say),
    ('SkillSpan',   skl),
    ('GREEN',       grn),
    ('GNEHM',       gnh),
    ('FIJO',        fij),
    ('Kompetencer', kom),
]

results = {}

for name, ds in EVAL_DATASETS:
    test_examples = list(ds['test'])
    random.shuffle(test_examples)
    test_examples = test_examples[:MAX_EXAMPLES]

    true_tags, pred_tags = [], []
    for example in tqdm(test_examples, desc=f'Evaluating {name}'):
        tokens, gold = get_tokens_and_tags(example)
        pred = predict(tokens)
        true_tags.append(gold)
        pred_tags.append(pred)

    f1 = f1_score(true_tags, pred_tags)
    results[name] = {'true': true_tags, 'pred': pred_tags, 'f1': f1}
    print(f'{name}: F1 = {f1:.4f}')

print('\nAll evaluations done.')

Evaluating Sayfullina:   0%|          | 0/500 [00:00<?, ?it/s]/usr/local/lib/python3.12/dist-packages/transformers/generation/configuration_utils.py:590: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.7` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/transformers/generation/configuration_utils.py:595: UserWarning: `do_sample` is set to `False`. However, `top_p` is set to `0.8` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_p`.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/transformers/generation/configuration_utils.py:612: UserWarning: `do_sample` is set to `False`. However, `top_k` is set to `20` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_k`.
  warnings.warn(
Evaluating Sayfullina: 100%|██████████|

Sayfullina: F1 = 0.6635


Evaluating SkillSpan: 100%|██████████| 500/500 [24:46<00:00,  2.97s/it]


SkillSpan: F1 = 0.3087


Evaluating GREEN: 100%|██████████| 335/335 [20:46<00:00,  3.72s/it]


...

GNEHM: F1 = 0.4894


Evaluating FIJO: 100%|██████████| 50/50 [06:05<00:00,  7.31s/it]


FIJO: F1 = 0.1951


Evaluating Kompetencer: 100%|██████████| 262/262 [22:11<00:00,  5.08s/it]

Kompetencer: F1 = 0.2169

All evaluations done.


In [12]:
# Cell 13 — Results report
paper_targets = {
    'Sayfullina':  0.6118,
    'SkillSpan':   0.4665,
    'GREEN':       0.3646,
    'GNEHM':       0.2224,
    'FIJO':        0.3532,
    'Kompetencer': 0.3921,
}

print('=' * 65)
print('Phase 6 — Combined SRICL: Qwen2.5-3B, all 6 datasets')
print('=' * 65)

for name, res in results.items():
    print(f'\n--- {name} ---')
    print(classification_report(res['true'], res['pred']))
    print(f'Strict F1:    {res["f1"]:.4f}')
    print(f'Paper target: {paper_targets.get(name, "?")}')

print('\n' + '=' * 65)
print('Summary')
print('=' * 65)
print(f'{"Dataset":<15} {"Our F1":>8} {"Paper F1":>10} {"Delta":>8}')
print('-' * 45)
f1_values = []
for name, res in results.items():
    our   = res['f1']
    paper = paper_targets.get(name, 0)
    delta = our - paper
    f1_values.append(our)
    print(f'{name:<15} {our:>8.4f} {paper:>10.4f} {delta:>+8.4f}')
print('-' * 45)
avg_our   = sum(f1_values) / len(f1_values)
avg_paper = sum(paper_targets.values()) / len(paper_targets)
print(f'{"Average":<15} {avg_our:>8.4f} {avg_paper:>10.4f} {avg_our - avg_paper:>+8.4f}')

Phase 6 — Combined SRICL: Qwen2.5-3B, all 6 datasets

--- Sayfullina ---
              precision    recall  f1-score   support

       SKILL       0.64      0.69      0.66       503

   micro avg       0.64      0.69      0.66       503
   macro avg       0.64      0.69      0.66       503
weighted avg       0.64      0.69      0.66       503

Strict F1:    0.6635
Paper target: 0.6118

--- SkillSpan ---
              precision    recall  f1-score   support

       SKILL       0.26      0.39      0.31       124

   micro avg       0.26      0.39      0.31       124
   macro avg       0.26      0.39      0.31       124
weighted avg       0.26      0.39      0.31       124

Strict F1:    0.3087
Paper target: 0.4665

--- GREEN ---
              precision    recall  f1-score   support

       SKILL       0.54      0.25      0.34       672

   micro avg       0.54      0.25      0.34       672
   macro avg       0.54      0.25      0.34       672
weighted avg       0.54      0.25      0.34  

In [20]:
say['train']

Dataset({
    features: ['idx', 'tokens', 'tags_skill'],
    num_rows: 3705
})

In [30]:
# Cell 12 — Evaluate all 6 datasets (10 examples each)
MAX_EXAMPLES = 10

EVAL_DATASETS = [
    ('Sayfullina',  say),
    ('SkillSpan',   skl),
    ('GREEN',       grn),
    ('GNEHM',       gnh),
    ('FIJO',        fij),
    ('Kompetencer', kom),
]

results = {}
rows = []
example_id = 0

for name, ds in EVAL_DATASETS:
    test_examples = list(ds['test'])
    random.shuffle(test_examples)
    test_examples = test_examples[:MAX_EXAMPLES]

    true_tags, pred_tags = [], []
    for example in tqdm(test_examples, desc=f'Evaluating {name}'):
        tokens, gold = get_tokens_and_tags(example)
        pred = predict(tokens)
        true_tags.append(gold)
        pred_tags.append(pred)

        for token_id, (token, gold_tag, pred_tag) in enumerate(zip(tokens, gold, pred)):
            rows.append({
                'dataset':    name,
                'example_id': example_id,
                'token_id':   token_id,
                'token':      token,
                'gold':       gold_tag,
                'pred':       pred_tag,
            })
        example_id += 1

    f1 = f1_score(true_tags, pred_tags)
    results[name] = {'true': true_tags, 'pred': pred_tags, 'f1': f1}
    print(f'{name}: F1 = {f1:.4f}')

df = pd.DataFrame(rows)
print(f'\nAll evaluations done. Token rows: {len(df)}')
df.head(20)


Evaluating Sayfullina: 100%|██████████| 10/10 [00:29<00:00,  2.91s/it]


Sayfullina: F1 = 0.6364


Evaluating SkillSpan: 100%|██████████| 10/10 [00:40<00:00,  4.05s/it]


SkillSpan: F1 = 0.2857


Evaluating GREEN: 100%|██████████| 10/10 [00:31<00:00,  3.16s/it]


...

Evaluating FIJO: 100%|██████████| 10/10 [01:23<00:00,  8.39s/it]


FIJO: F1 = 0.1200


Evaluating Kompetencer: 100%|██████████| 10/10 [00:33<00:00,  3.36s/it]

Kompetencer: F1 = 0.3333

All evaluations done. Token rows: 1194


,dataset,example_id,token_id,token,gold,pred
0,Sayfullina,0,0,the,O,O
1,Sayfullina,0,1,ability,B-SKILL,O
2,Sayfullina,0,2,to,I-SKILL,O
3,Sayfullina,0,3,negotiate,I-SKILL,B-SKILL
4,Sayfullina,0,4,high,O,O
5,Sayfullina,0,5,value,O,O
6,Sayfullina,0,6,",",O,O
7,Sayfullina,0,7,complex,O,O
8,Sayfullina,0,8,system,O,O
9,Sayfullina,0,9,on,O,O


In [33]:
df.to_csv('examples.csv')